# FitCheck 상의 합성 확인

인물 사진 + 상의 상품 사진 → FASHN 기본 합성 결과를 확인합니다.
기준 옷 치수·기준점·00_ME 사진 없이 사용할 수 있습니다. 사이즈별 치수 조정은 기존 디버거에서 확인합니다.

먼저 **런타임 → 런타임 유형 변경 → GPU(T4)**를 선택하고, 아래 셀을 차례대로 실행하세요.
마지막 셀에서 나오는 `gradio.live` 링크를 열어 사진 두 장을 올리고 **합성하기**를 누르세요.
첫 합성은 모델 로딩 시간이 추가되며, 같은 서버에서 다음 합성은 로드한 모델을 재사용합니다.

In [ ]:
# 1. GPU 확인
import subprocess
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], check=True)

In [ ]:
# 2. 현재 FitCheck 코드와 FASHN 받기
import os
import sys
from pathlib import Path

APP = '/content/fitcheck-synthesis'
FASHN = '/content/fashn-vton-1.5'
WEIGHTS = '/content/fashn-weights'
REPOSITORY = 'https://github.com/eunzae02-glitch/virtual-fitting-app-master.git'

if not Path(APP).exists():
    subprocess.run(['git', 'clone', '--branch', 'main', REPOSITORY, APP], check=True)
else:
    origin = subprocess.check_output(['git', '-C', APP, 'remote', 'get-url', 'origin'], text=True).strip()
    if origin.rstrip('/').removesuffix('.git') != REPOSITORY.removesuffix('.git'):
        raise RuntimeError('APP 경로에 다른 저장소가 있습니다. APP를 새 폴더 경로로 바꿔주세요.')
    subprocess.run(['git', '-C', APP, 'pull', '--ff-only'], check=True)
if not Path(FASHN).exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/fashn-AI/fashn-vton-1.5.git', FASHN], check=True)
if not (Path(APP) / 'scripts' / 'debug_synthesis.py').is_file():
    raise RuntimeError('debug_synthesis.py가 없습니다. PC의 변경사항을 GitHub에 Push한 뒤 다시 실행해주세요.')
print('코드 준비 완료')

In [ ]:
# 3. 합성 엔진과 웹 화면 설치
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', FASHN, 'gradio>=5,<7', 'Pillow>=10,<13'], check=True)
print('설치 완료')

In [ ]:
# 4. 합성·자세 가중치 준비 (이미 받은 파일은 다운로드 캐시 재사용)
subprocess.run([sys.executable, f'{FASHN}/scripts/download_weights.py', '--weights-dir', WEIGHTS], check=True)
print('가중치 준비 완료')

In [ ]:
# 5. 합성 확인 화면 열기 — 출력의 gradio.live 링크를 클릭하세요.
# 화면을 사용하는 동안 이 셀은 실행 중입니다. 종료할 때 중지 버튼을 누르세요.
os.environ['FASHN_REPO'] = FASHN
os.environ['FASHN_WEIGHTS'] = WEIGHTS
subprocess.run([sys.executable, '-u', f'{APP}/scripts/debug_synthesis.py', '--share'], check=True)